# Chapitre 3 : Exploration et nettoyage des données

*MLSMM2151 Data Science · Séance 2, vendredi 27 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 2 sur 4 : Manquants, doublons, valeurs aberrantes

**Durée estimée : 20 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- lire un tableau de valeurs manquantes et dire, pour chaque colonne, si
  l'absence est un hasard ou une information ;
- choisir entre supprimer, imputer et ajouter un indicateur de manque, et
  justifier ce choix ;
- récupérer une information manquante dans une autre colonne avec
  `str.extract` ;
- repérer doublons exacts et quasi-doublons, et décider de leur sort selon la
  question posée ;
- comparer une règle statistique et une règle métier pour écarter des valeurs
  aberrantes.

Ce notebook repart du fichier brut. Il n'a besoin que d'une conversion du
notebook 1, celle du prix : on la refait en une cellule.

In [ ]:
# Cette ligne récupère le fichier utils/donnees.py du dépôt du cours.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

import pandas as pd
import matplotlib.pyplot as plt
from donnees import charger

In [ ]:
annonces = charger("airbnb_raw")

In [ ]:
# La conversion du notebook 1 : "$1,234.00" devient 1234.0.
annonces["prix"] = (
    annonces["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

## 3.3. Gestion des valeurs manquantes

Une valeur manquante n'est pas une case vide qu'il faut remplir. C'est une
question : **pourquoi** cette information n'est-elle pas là ? La réponse
commande le traitement. Commençons par la vue d'ensemble.

In [ ]:
# Proportion de manquants par colonne, des plus touchées aux moins touchées.
taux_manquants = annonces.isna().mean().sort_values(ascending=False)
taux_manquants[taux_manquants > 0].round(3)

**Lecture.** Des colonnes manquent **ensemble**, au même taux, et c'est le
premier indice à relever.

- `license` : 100 %. Aucune information, la colonne se supprime sans débat.
- Le taux et le délai de réponse de l'hôte : 19,3 %, ensemble.
- Le prix (`price` et notre `prix`) et le revenu estimé : 15,2 %, ensemble,
  parce que le revenu est calculé à partir du prix. Le prix manquant fait
  l'objet de l'atelier du chapitre 4, nous n'y touchons pas ici.
- `beds` et `bathrooms`, autour de 15 % : nous revenons à `bathrooms` plus bas.
- **Sept colonnes d'avis manquent exactement au même taux, 13,8 %.** Ce n'est
  pas une coïncidence.

Avant d'aller plus loin, voici le réflexe à ne **pas** avoir.

In [ ]:
# « Supprimons les lignes incomplètes. »
annonces.dropna().shape

Zéro ligne. `dropna()` supprime toute ligne qui a au moins un manquant, et
`license` est vide partout : il ne reste rien. Sur un fichier réel, supprimer
à l'aveugle revient souvent à jeter presque tout, ou pire, à jeter une partie
seulement, toujours la même, sans s'en rendre compte.

### Des manquants qui ne sont pas un hasard

Les sept colonnes d'avis manquent ensemble. Hypothèse : ce sont les annonces
qui n'ont **jamais reçu d'avis**. Une hypothèse se vérifie.

In [ ]:
sans_note = annonces["review_scores_rating"].isna()
zero_avis = annonces["number_of_reviews"] == 0
pd.crosstab(sans_note, zero_avis, rownames=["note manquante"], colnames=["zéro avis"])

Le tableau croisé est sans appel : les 925 annonces sans note sont exactement
les 925 annonces à zéro avis. L'absence de note n'est pas un trou dans les
données, **c'est une information** : l'annonce est nouvelle, ou personne n'y a
séjourné. On parle de manquant **non aléatoire**.

### Trois stratégies

| Stratégie | Quand l'utiliser | Le risque |
|---|---|---|
| **Supprimer** la ligne ou la colonne | Colonne vide (`license`), ou très peu de lignes touchées, au hasard | Perdre des lignes qui ne ressemblent pas aux autres |
| **Imputer** : remplacer par une valeur (médiane, mode) | Manquant aléatoire, peu fréquent | Inventer une information, et effacer le fait qu'elle manquait |
| **Ajouter un indicateur** de manque | Le manque lui-même a un sens | Aucun, si l'on n'oublie pas de traiter la colonne d'origine |

Voici ce que donne une imputation naïve sur la note.

In [ ]:
mediane_note = annonces["review_scores_rating"].median()
note_imputee = annonces["review_scores_rating"].fillna(mediane_note)
print(f"Médiane des notes : {mediane_note}")
print(f"Annonces jamais notées qui reçoivent cette note : {sans_note.sum()}")

925 annonces que personne n'a notées reçoivent la note médiane, 4,8 sur 5,
comme si chacune était une annonce typique, appréciée de ses voyageurs. C'est
une invention, et elle efface l'information « jamais notée ». Ici, la bonne réponse est
l'indicateur : une colonne qui dit si l'annonce a des avis, à côté de la note.

In [ ]:
annonces["a_des_avis"] = (annonces["number_of_reviews"] > 0).astype(int)
annonces["a_des_avis"].value_counts()

> **Un problème pour plus tard.** Cette médiane a été calculée sur **tout** le
> fichier. Le jour où nous séparerons des annonces d'entraînement et des
> annonces de test, une médiane calculée sur tout le fichier aura « vu » les
> annonces de test. C'est une fuite de données, et c'est l'objet du chapitre 5.
> Retenez seulement ceci pour l'instant : imputer n'est jamais neutre.

### Une information qui était ailleurs

Dernier cas, le plus simple et le plus souvent raté : la valeur manque dans
une colonne, mais elle est écrite dans une autre.

In [ ]:
sdb_recuperables = annonces["bathrooms"].isna() & annonces["bathrooms_text"].notna()
print(f"{sdb_recuperables.sum()} annonces sans `bathrooms`, mais avec `bathrooms_text`")

In [ ]:
annonces["bathrooms_text"].value_counts()

991 annonces ont un nombre de salles de bain manquant alors que le texte est
là. Imputer une médiane serait absurde : il suffit de lire le texte. La plupart
des valeurs commencent par un nombre, `1 bath`, `1.5 shared baths`. Trois ne
contiennent aucun chiffre : `Half-bath`, `Shared half-bath`,
`Private half-bath`.

### Hack-Time 🛠️

Nous venons de voir que 991 nombres de salles de bain manquants sont écrits en
toutes lettres dans `bathrooms_text`. À votre tour, dans la cellule
ci-dessous :

1. Extrayez le nombre du texte. Une piste :
   `annonces["bathrooms_text"].str.extract(r"(\d+\.?\d*)")[0].astype(float)`.
   L'expression régulière lit « un ou plusieurs chiffres, éventuellement un
   point et d'autres chiffres ».
2. Les `half-bath` n'ont pas de chiffre : donnez-leur 0,5. Une piste :
   `.str.contains("half", case=False, na=False)` puis `.mask(demi, 0.5)`.
3. Complétez `bathrooms` avec `fillna`, dans une nouvelle colonne
   `salles_de_bain`, et comptez les manquants qui restent.

**Cinq minutes.**

**Défi optionnel**
Sur les annonces où `bathrooms` **et** le texte sont remplis, votre extraction
donne-t-elle le même nombre ? Si oui, qu'est-ce que cela vous apprend sur la
façon dont Inside Airbnb a construit `bathrooms` ? Et quelle information du
texte votre colonne a-t-elle perdue ?

In [ ]:
# Votre code ici

## 3.4. Détection et traitement des doublons

Une annonce comptée deux fois pèse deux fois dans une moyenne, et dans un
modèle. Premier contrôle : les doublons **exacts**, puis l'identifiant.

In [ ]:
print(f"Lignes entièrement identiques : {annonces.duplicated().sum()}")
print(f"Identifiants répétés          : {annonces['id'].duplicated().sum()}")

Aucun. Le fichier est-il propre pour autant ? Un doublon exact est rare : le
même logement publié deux fois aura deux identifiants, et sans doute un nombre
d'avis différent. Cherchons plutôt des **quasi-doublons** : même hôte, même
titre.

In [ ]:
cle = ["host_id", "name"]
quasi = annonces.duplicated(subset=cle, keep=False)
print(f"{quasi.sum()} annonces partagent leur titre et leur hôte avec une autre")
print(f"{annonces.duplicated(subset=cle).sum()} seraient supprimées en ne gardant que la première")

In [ ]:
# Les groupes les plus gros.
annonces[quasi].groupby(cle).size().sort_values(ascending=False).head()

In [ ]:
colonnes = ["id", "room_type", "accommodates", "neighbourhood_cleansed", "latitude", "longitude", "number_of_reviews", "prix"]
annonces.loc[annonces["name"] == "Luxurious studio lively area", colonnes]

**Lecture.** 176 annonces, en 64 groupes, partagent leur titre et leur hôte
avec une autre. Le plus gros groupe compte 10 annonces « Luxurious studio
lively area », du même hôte. Mais regardez-les : identifiants différents,
capacités de 2 ou de 4 personnes, nombres d'avis différents, positions voisines
mais pas identiques (Inside Airbnb bruite légèrement les coordonnées), entre
Anderlecht et Molenbeek-Saint-Jean.

Ce sont très probablement **dix chambres réelles** d'un même immeuble, louées
sous un titre commun. Pas des erreurs de saisie. La question n'est donc pas
« est-ce un doublon ? » mais « que veut-on compter ? ».

### Hack-Time 🛠️

Nous venons de trouver 112 annonces qui seraient supprimées par un
dédoublonnage sur `host_id` et `name`, dont dix chambres d'un même hôte. Dans
la cellule ci-dessous, dites si vous les gardez ou les retirez pour chacune
de ces trois questions :

1. « Combien de logements Airbnb propose-t-il à Molenbeek-Saint-Jean ? »
2. « Quel prix afficher pour un nouveau logement ? », un modèle de prédiction.
3. « Quelle part du marché bruxellois est tenue par les hôtes qui gèrent plus
   de dix annonces ? »

Une phrase de justification par question. La réponse n'est pas la même pour
les trois.

**Trois minutes.**

**Défi optionnel**
Une onzième annonce de cet hôte s'appelle « Luxurious studio lively area. »,
avec un point final. Retrouvez-la avec
`annonces["name"].str.contains("Luxurious studio lively area")`. Qu'est-ce
que cela dit de la règle « même titre » ? Comment la rendre plus robuste, et
que risquez-vous en la rendant trop souple ?

*Vos réponses :*

1.
2.
3.

## 3.5. Détection et traitement des valeurs aberrantes

Une valeur aberrante est une valeur très éloignée des autres. Ce n'est pas
forcément une erreur : un château de vingt chambres existe. La question est
toujours double : **est-elle vraie ?** et **appartient-elle à la population
que j'étudie ?**

In [ ]:
annonces[["prix", "minimum_nights", "maximum_nights", "beds"]].describe().round(1)

In [ ]:
# Le prix le plus élevé : qui est-ce ?
annonces.loc[annonces["prix"] == annonces["prix"].max(), ["name", "neighbourhood_cleansed", "room_type", "accommodates", "prix", "number_of_reviews"]]

Un appartement de vacances à Woluwe-Saint-Lambert à **9 999** par nuit. Le
chiffre lui-même trahit l'intention : c'est le plus grand nombre à quatre
chiffres, un prix dissuasif posé par un hôte qui ne veut pas, ou plus, être
réservé. La valeur est réelle, au sens où elle est bien affichée. Elle ne
décrit pas le marché.

In [ ]:
# Les séjours minimum les plus longs.
annonces["minimum_nights"].value_counts().sort_index().tail(5)

In [ ]:
annonces.loc[annonces["minimum_nights"] == 500, ["host_id", "name", "room_type", "prix"]]

Trois chambres près de la gare du Midi, même hôte, entre 16 et 18 par nuit,
exigent un séjour minimum de 500 nuits. Ce ne sont plus des locations touristiques, mais des baux déguisés
ou des annonces gelées.

In [ ]:
# Les séjours maximum les plus fréquents.
annonces["maximum_nights"].value_counts().head(3)

1 125 nuits pour 1 292 annonces : ce n'est pas un choix de 1 292 hôtes, c'est
**la valeur par défaut** d'Airbnb, qui signifie « pas de maximum ». On appelle
cela une **valeur sentinelle** : un nombre qui code une absence. Le traiter
comme une durée réelle, par exemple en calculant une moyenne, serait une
erreur. Une sentinelle n'est pas une aberrante, c'est un manquant déguisé.

### Deux règles pour décider

Pour écarter les valeurs extrêmes, il existe deux familles de règles.

- **La règle statistique de l'écart interquartile (IQR).** On calcule
  Q1 et Q3, le premier et le troisième quartile, puis l'écart
  IQR = Q3 - Q1. Est aberrante toute valeur en dessous de Q1 - 1,5 × IQR ou
  au-dessus de Q3 + 1,5 × IQR. Elle ne demande aucune connaissance du domaine,
  et c'est sa force comme sa faiblesse.
- **La règle métier.** On fixe des bornes qui ont un sens pour la question
  posée. Pour construire la table de modélisation `airbnb_train` que vous
  utiliserez à partir du chapitre 5, nous avons gardé les prix **entre 20 et
  1 000** par nuit, et les séjours minimum d'au plus 365 nuits.

In [ ]:
# La règle métier, appliquée aux annonces qui ont un prix.
prix = annonces["prix"].dropna()
hors_bornes_metier = ~prix.between(20, 1000)
print(f"Règle métier [20, 1000] : {hors_bornes_metier.sum()} annonces écartées sur {len(prix)}")

### Hack-Time 🛠️

Nous venons d'appliquer la règle métier : elle écarte 15 annonces. Comparons
avec la règle statistique. Dans la cellule ci-dessous, sur la même série
`prix` :

1. Calculez Q1, Q3 et l'écart IQR. Une piste : `prix.quantile(0.25)`.
2. Calculez les deux bornes, et comptez les annonces en dehors.
3. En une phrase : quel genre de logement la règle IQR écarte-t-elle, que la
   règle métier garde ?

**Trois minutes.**

**Défi optionnel**
Les prix sont très asymétriques : beaucoup de petits, quelques très grands.
Refaites la règle IQR sur `np.log(prix)` (après `import numpy as np`), puis
revenez aux euros avec `np.exp` pour lire les bornes. Que change le passage
au logarithme ?

In [ ]:
# Votre code ici

## À retenir

- Un manquant est une question : pourquoi l'information n'est-elle pas là ?
  Les 925 notes manquantes sont exactement les annonces sans avis, et ce
  manque est une information.
- Trois stratégies : supprimer, imputer, ajouter un indicateur. `dropna()` à
  l'aveugle peut tout supprimer, et imputer invente toujours quelque chose.
- Avant d'imputer, cherchez si l'information n'est pas ailleurs :
  `bathrooms_text` contenait 991 valeurs « manquantes ».
- Aucun doublon exact ne veut pas dire aucun doublon. Garder ou retirer des
  quasi-doublons dépend de la question posée.
- Une valeur extrême peut être vraie, fausse, ou une sentinelle comme 1 125.
  La règle IQR ignore le contexte, la règle métier l'écrit : dans les deux
  cas, la règle se justifie par écrit.

**La suite** : le notebook 3 traite les trois colonnes de dates, que nous
avons jusqu'ici converties sans les exploiter : en extraire une année, un jour
de la semaine, une durée.